In [3]:
import os

from dotenv import load_dotenv
_ = load_dotenv()

TPCDS_TOOLS_DIR = os.getenv("TPCDS_TOOLS_DIR")
DATA_DIR = os.path.join(TPCDS_TOOLS_DIR, "data")

print(f"TPCDS_TOOLS_DIR: {TPCDS_TOOLS_DIR}")
print(f"DATA_DIR: {DATA_DIR}")

TPCDS_TOOLS_DIR: /home/basbenja/Documents/DEAI/data-warehouses/project/part1/tpcds-tools
DATA_DIR: /home/basbenja/Documents/DEAI/data-warehouses/project/part1/tpcds-tools/data


In [7]:
sql_schema_path = os.path.join(TPCDS_TOOLS_DIR, "tpcds.sql")
with open(sql_schema_path, "r") as f:
    sql_schema = f.read()

no_comments = "\n".join(
    line for line in sql_schema.splitlines() if not line.strip().startswith("--")
)
tables = [table.strip() for table in no_comments.split(";") if table.strip()]

for t in tables:
    print(t)

create table dbgen_version
(
    dv_version                varchar(16)                   ,
    dv_create_date            date                          ,
    dv_create_time            time                          ,
    dv_cmdline_args           varchar(200)                  
)
create table customer_address
(
    ca_address_sk             integer               not null,
    ca_address_id             char(16)              not null,
    ca_street_number          char(10)                      ,
    ca_street_name            varchar(60)                   ,
    ca_street_type            char(15)                      ,
    ca_suite_number           char(10)                      ,
    ca_city                   varchar(60)                   ,
    ca_county                 varchar(30)                   ,
    ca_state                  char(2)                       ,
    ca_zip                    char(10)                      ,
    ca_country                varchar(20)                   ,
    ca_g

In [2]:
print(os.listdir(DATA_DIR))

['household_demographics.dat', 'warehouse.dat', 'web_page.dat', 'income_band.dat', 'web_site.dat', 'ship_mode.dat', 'web_sales.dat', 'date_dim.dat', 'inventory.dat', 'dbgen_version.dat', 'item.dat', 'catalog_sales.dat', 'customer.dat', 'store_sales.dat', 'web_returns.dat', 'promotion.dat', 'catalog_returns.dat', 'customer_demographics.dat', 'call_center.dat', 'store.dat', 'reason.dat', 'time_dim.dat', 'customer_address.dat', 'store_returns.dat', 'catalog_page.dat']


## **Starting Point: SparkSession**
The entry point into all functionality in Spark is the `SparkSession` class. To create a basic `SparkSession`, just use `SparkSession.builder`.

In [3]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("tpcds") \
    .master("local[*]") \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/09/21 15:30:22 WARN Utils: Your hostname, bbas, resolves to a loopback address: 127.0.1.1; using 10.66.79.98 instead (on interface wlp0s20f3)
26/09/21 15:30:22 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/21 15:30:24 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [11]:
path = "/home/basbenja/Documents/DEAI/data-warehouses/project/part1/tpcds-tools/tpcds.sql"
with open(path, "r") as f:
    sql = f.read()
    spark.sql(sql)

ParseException: 
[PARSE_SYNTAX_ERROR] Syntax error at or near 'create': extra input 'create'. SQLSTATE: 42601 (line 44, pos 0)


In [13]:
spark.sql("""create table dbgen_version
(
    dv_version                varchar(16)                   ,
    dv_create_date            date                          ,
    dv_create_time            time                          ,
    dv_cmdline_args           varchar(200)                  
);""")

AnalysisException: [UNSUPPORTED_TIME_TYPE] The data type TIME is not supported. SQLSTATE: 0A000

In [10]:
df = spark.read.csv(
    path=os.path.join(DATA_DIR, "warehouse.dat"),
    sep='|'
)
print(type(df))

<class 'pyspark.sql.classic.dataframe.DataFrame'>
